# NB10: Transport & Unmapped Reaction Analysis

**Purpose**: Characterize the role of transport reactions in the Rosetta mapping gap
and explore text-based matching as an alternative pathway for unmapped transport reactions.
Then characterize the remaining unmapped non-transport reactions.

**Context**: 16,992 of 34,343 balanced reactions are unmapped (49.5%). Transport
reactions are expected to be disproportionately unmapped because they often lack
EC numbers. After accounting for transport, ~12,000 non-transport reactions remain
unmapped — we decompose those by EC availability, database origin, and thermodynamics.

**Requires**: BERDL JupyterHub (Spark session for UniProt name table query)

**Output**: `transport_analysis.parquet`, `uniprot_transport_proteins.parquet`,
`transport_text_match_candidates.parquet`, `transport_breakdown.png`, `unmapped_characterization.png`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import gc

DATA_DIR = '../data'
FIG_DIR = '../figures'
USER_DIR = '../user_data'

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,
})

## 1. Transport vs Mapped Cross-Tabulation

In [2]:
rxns = pd.read_csv(f'{DATA_DIR}/reactions_all.tsv', sep='\t',
                    usecols=['id', 'is_transport', 'status', 'name', 'abbreviation', 'deltag'])
rxns = rxns[rxns['status'] == 'OK'].copy()
rxns['rxn_bare'] = rxns['id'].str.replace('seed.reaction:', '', regex=False)
print(f'Balanced reactions: {len(rxns):,}')
print(f'Transport: {rxns["is_transport"].sum():,}')
print(f'Non-transport: {(~rxns["is_transport"]).sum():,}')

evidence = pd.read_parquet(f'{DATA_DIR}/evidence_integration_summary.parquet',
                           columns=['rxn_bare', 'any_evidence', 'confidence', 'n_channels'])

merged = rxns.merge(evidence, on='rxn_bare', how='left')
merged['any_evidence'] = merged['any_evidence'].fillna(False)
merged['mapped'] = merged['any_evidence'].map({True: 'Mapped', False: 'Unmapped'})

ct = pd.crosstab(merged['is_transport'], merged['mapped'], margins=True)
ct.index = ct.index.map({False: 'Non-transport', True: 'Transport', 'All': 'All'})
print(f'\nCross-tabulation:')
print(ct.to_string())

unmapped_transport = ct.loc['Transport', 'Unmapped']
unmapped_total = ct.loc['All', 'Unmapped']
mapped_transport = ct.loc['Transport', 'Mapped']
mapped_total = ct.loc['All', 'Mapped']
total_transport = ct.loc['Transport', 'All']

print(f'\nTransport reactions are {100*unmapped_transport/total_transport:.1f}% unmapped '
      f'vs {100*ct.loc["Non-transport", "Unmapped"]/ct.loc["Non-transport", "All"]:.1f}% for non-transport')
print(f'Transport as % of unmapped: {100*unmapped_transport/unmapped_total:.1f}%')
print(f'Transport as % of mapped: {100*mapped_transport/mapped_total:.1f}%')

print(f'\nConfidence breakdown for transport vs non-transport (mapped only):')
mapped_only = merged[merged['any_evidence']]
conf_ct = pd.crosstab(mapped_only['is_transport'], mapped_only['confidence'])
conf_ct.index = conf_ct.index.map({False: 'Non-transport', True: 'Transport'})
print(conf_ct.to_string())

Balanced reactions: 34,343
Transport: 6,004
Non-transport: 28,339

Cross-tabulation:
mapped         Mapped  Unmapped    All
is_transport                          
Non-transport   16301     12038  28339
Transport        1050      4954   6004
All             17351     16992  34343

Transport reactions are 82.5% unmapped vs 42.5% for non-transport
Transport as % of unmapped: 29.2%
Transport as % of mapped: 6.1%

Confidence breakdown for transport vs non-transport (mapped only):
confidence      high  low  medium
is_transport                     
Non-transport  13752  116    2433
Transport        718   13     319


## 2. Transport Breakdown Figure

In [3]:
mapped_nt = ct.loc['Non-transport', 'Mapped']
mapped_t = ct.loc['Transport', 'Mapped']
unmapped_nt = ct.loc['Non-transport', 'Unmapped']
unmapped_t = ct.loc['Transport', 'Unmapped']

fig, ax = plt.subplots(figsize=(8, 5))
bars_bottom = [mapped_nt, unmapped_nt]
bars_top = [mapped_t, unmapped_t]
x = [0, 1]
labels = ['Mapped', 'Unmapped']

b1 = ax.bar(x, bars_bottom, 0.5, label='Non-transport', color='#4c78a8')
b2 = ax.bar(x, bars_top, 0.5, bottom=bars_bottom, label='Transport', color='#f58518')

for i, (nt, t) in enumerate(zip(bars_bottom, bars_top)):
    total = nt + t
    ax.text(i, nt/2, f'{nt:,}\n({100*nt/total:.0f}%)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    ax.text(i, nt + t/2, f'{t:,}\n({100*t/total:.0f}%)', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=13)
ax.set_ylabel('Number of Balanced Reactions')
ax.set_title('Transport Reactions in the Rosetta Mapping Gap')
ax.legend(loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

fig.savefig(f'{FIG_DIR}/transport_breakdown.png')
plt.show()
print('Saved: transport_breakdown.png')

Saved: transport_breakdown.png


In [4]:
transport_rxns = merged[merged['is_transport']].copy()
unmapped_transport_rxns = transport_rxns[~transport_rxns['any_evidence']]

has_name = unmapped_transport_rxns['name'].notna().sum()
no_name = unmapped_transport_rxns['name'].isna().sum()
print(f'Unmapped transport reactions: {len(unmapped_transport_rxns):,}')
print(f'  With name: {has_name:,}')
print(f'  No name:   {no_name:,}')

print(f'\nSample unmapped transport reaction names:')
samples = unmapped_transport_rxns[unmapped_transport_rxns['name'].notna()]['name'].sample(20, random_state=42)
for i, name in enumerate(samples, 1):
    print(f'  {i:2d}. {name}')

merged.to_parquet(f'{DATA_DIR}/transport_analysis.parquet', index=False)
print(f'\nSaved: transport_analysis.parquet ({len(merged):,} rows)')

Unmapped transport reactions: 4,954
  With name: 4,572
  No name:   382

Sample unmapped transport reaction names:
   1. TRANS-RXNBWI-115637.ce.maizeexp.PHE_PHE
   2. rxn11100
   3. TRANS-RXNBWI-115637.ce.maizeexp.THR_THR
   4. m-xylene  outer membrane porin transport
   5. Guanosine 3'-phsophate transport in/out via proton symport
   6. sphinganine 1-phosphate endoplasmic reticular transport
   7. Cotransport of L-glutamate and H+
   8. 29139
   9. transport of 4-coumarate [extraorganism-cytosol](secondary symport)
  10. -
  11. Glucose-6-phosphate transport via phosphate antiport (periplasm)
  12. spermine transport via proton antiport irreversible
  13. TRANS-RXNBWI-115525.ce.maizeexp.NA+_NA+
  14. TRANS-RXNAVI-26732.ce.brachyexp.SORBITOL_SORBITOL
  15. Taurocholic Acid transport in via proton symport
  16. Urea active transporter
  17. TRANS-RXNAVI-26433.ce.brachyexp.TAGATOSE_TAGATOSE
  18. thiamine transport via ABC system
  19. -
  20. choline transport via ABC system (periplasm)

## 3. UniProt Transport Proteins

In [5]:
import sys
sys.path.insert(0, '../../scripts')
from berdl_notebook_utils.setup_spark_session import get_spark_session
import pyarrow as pa
import pyarrow.parquet as pq

spark = get_spark_session()

In [6]:
transport_proteins_df = spark.sql("""
    SELECT REPLACE(entity_id, 'uniprot:', '') AS protein,
           name, description
    FROM refdata_uniprot.name
    WHERE LOWER(description) LIKE '%transport%'
       OR LOWER(description) LIKE '%permease%'
       OR LOWER(description) LIKE '%symport%'
       OR LOWER(description) LIKE '%antiport%'
       OR LOWER(description) LIKE '%efflux%'
       OR LOWER(description) LIKE '%porin%'
       OR LOWER(name) LIKE '%transport%'
       OR LOWER(name) LIKE '%permease%'
""").toPandas()

print(f'Transport-related protein entries: {len(transport_proteins_df):,}')
print(f'Unique proteins: {transport_proteins_df["protein"].nunique():,}')
print(f'\nSample entries:')
print(transport_proteins_df.head(10).to_string(index=False))

Transport-related protein entries: 11,392,424


Unique proteins: 10,440,445

Sample entries:
   protein                                          name                   description
A0A374NVG6          PTS fructose transporter subunit IIA   UniProt submitted full name
A0A100VS83      Arabinogalactan ABC transporter permease   UniProt submitted full name
A0A100VS83                Sugar ABC transporter permease   UniProt submitted full name
A0A7R9L8Q2        Sodium-dependent glucose transporter 1 UniProt recommended full name
A0AAW6UYZ0       CorA family divalent cation transporter   UniProt submitted full name
A0AAW6UYZ0   Magnesium and cobalt transport protein CorA   UniProt submitted full name
A0A060ZUP5      Tricarboxylic transport membrane protein   UniProt submitted full name
A0A7X3H6G7         Probable membrane transporter protein UniProt recommended full name
A0AAX2IJX2 Lipoprotein-releasing system permease protein   UniProt submitted full name
    X5JZZ7                        Riboflavin transporter UniProt recommended full nam

In [7]:
tier1_proteins = set(
    pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet',
                    columns=['protein'])['protein']
)
print(f'Tier 1 unique proteins: {len(tier1_proteins):,}')

transport_proteins_df['in_tier1'] = transport_proteins_df['protein'].isin(tier1_proteins)
tier1_transport = transport_proteins_df[transport_proteins_df['in_tier1']]
print(f'Transport proteins in Tier 1: {tier1_transport["protein"].nunique():,}')

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
transport_rxn_ids = set(merged[merged['is_transport'] & merged['any_evidence']]['rxn_bare'])
transport_ecs = set(ec_bridge[ec_bridge['rxn_bare'].isin(transport_rxn_ids)]['ec'])

tier1_ec = pd.read_parquet(f'{DATA_DIR}/uniprot_native_protein_ec.parquet',
                           columns=['protein', 'ec'])
proteins_with_transport_ec = set(tier1_ec[tier1_ec['ec'].isin(transport_ecs)]['protein'])
del tier1_ec
gc.collect()

tier1_transport_unique = set(tier1_transport['protein'])
transport_no_current_map = tier1_transport_unique - proteins_with_transport_ec
print(f'\nTier 1 transport proteins without current transport reaction mapping: {len(transport_no_current_map):,}')
print(f'Tier 1 transport proteins with current transport reaction mapping: {len(tier1_transport_unique & proteins_with_transport_ec):,}')

table = pa.table({
    'protein': transport_proteins_df['protein'].tolist(),
    'name': transport_proteins_df['name'].tolist(),
    'description': transport_proteins_df['description'].tolist(),
    'in_tier1': transport_proteins_df['in_tier1'].tolist(),
})
pq.write_table(table, f'{DATA_DIR}/uniprot_transport_proteins.parquet')
del table
print(f'\nSaved: uniprot_transport_proteins.parquet ({len(transport_proteins_df):,} rows)')

del tier1_proteins, proteins_with_transport_ec
gc.collect()

Tier 1 unique proteins: 26,549,024


Transport proteins in Tier 1: 181,781



Tier 1 transport proteins without current transport reaction mapping: 16,332
Tier 1 transport proteins with current transport reaction mapping: 165,449



Saved: uniprot_transport_proteins.parquet (11,392,424 rows)


0

## 4. Text-Based Matching: Transport Proteins to Transport Reactions

In [8]:
STOP_WORDS = {'the', 'of', 'and', 'a', 'an', 'in', 'to', 'for', 'by', 'with',
              'is', 'at', 'or', 'via', 'from', 'into', 'out', 'no', 'not',
              'transport', 'transporter', 'transported', 'transporting',
              'permease', 'protein', 'putative', 'probable', 'predicted',
              'uncharacterized', 'family', 'member', 'like', 'related',
              'system', 'component', 'subunit', 'precursor', 'homolog',
              'reaction', 'exchange', 'diffusion', 'h+', 'h', 'proton'}

def extract_keywords(text):
    if pd.isna(text):
        return set()
    tokens = re.findall(r'[a-zA-Z0-9][-a-zA-Z0-9]*', text.lower())
    return {t for t in tokens if t not in STOP_WORDS and len(t) > 2}

unmapped_t = merged[(merged['is_transport']) & (~merged['any_evidence']) & (merged['name'].notna())].copy()
unmapped_t['rxn_keywords'] = unmapped_t['name'].apply(extract_keywords)
print(f'Unmapped transport reactions with names: {len(unmapped_t):,}')

tier1_tp_desc = tier1_transport[['protein', 'description']].drop_duplicates(subset=['protein'])
tier1_tp_desc = tier1_tp_desc[tier1_tp_desc['protein'].isin(transport_no_current_map)]
tier1_tp_desc['prot_keywords'] = tier1_tp_desc['description'].apply(extract_keywords)
print(f'Transport proteins for matching: {len(tier1_tp_desc):,}')

prot_keyword_index = {}
for _, row in tier1_tp_desc.iterrows():
    for kw in row['prot_keywords']:
        prot_keyword_index.setdefault(kw, []).append(row['protein'])

matches = []
for _, rxn_row in unmapped_t.iterrows():
    rxn_kws = rxn_row['rxn_keywords']
    if not rxn_kws:
        continue
    candidate_proteins = {}
    for kw in rxn_kws:
        for prot in prot_keyword_index.get(kw, []):
            candidate_proteins[prot] = candidate_proteins.get(prot, 0) + 1
    for prot, overlap in candidate_proteins.items():
        if overlap >= 2:
            matches.append({
                'rxn_bare': rxn_row['rxn_bare'],
                'rxn_name': rxn_row['name'],
                'protein': prot,
                'keyword_overlap': overlap,
                'shared_keywords': rxn_kws & set(tier1_tp_desc[tier1_tp_desc['protein'] == prot]['prot_keywords'].iloc[0])
            })

match_df = pd.DataFrame(matches)
if len(match_df) > 0:
    match_df['shared_keywords'] = match_df['shared_keywords'].apply(lambda s: ', '.join(sorted(s)))
    match_df = match_df.sort_values('keyword_overlap', ascending=False)
    print(f'\nCandidate matches (>=2 keyword overlap): {len(match_df):,}')
    print(f'Unique reactions matched: {match_df["rxn_bare"].nunique():,} / {len(unmapped_t):,}')
    print(f'Unique proteins involved: {match_df["protein"].nunique():,}')
    print(f'\nTop 15 matches by keyword overlap:')
    print(match_df.head(15)[['rxn_bare', 'rxn_name', 'protein', 'keyword_overlap', 'shared_keywords']].to_string(index=False))
    match_df.to_parquet(f'{DATA_DIR}/transport_text_match_candidates.parquet', index=False)
    print(f'\nSaved: transport_text_match_candidates.parquet ({len(match_df):,} rows)')
else:
    print('\nNo matches found with >=2 keyword overlap. Transport reaction names may be too generic.')
    pd.DataFrame(columns=['rxn_bare','rxn_name','protein','keyword_overlap','shared_keywords']).to_parquet(
        f'{DATA_DIR}/transport_text_match_candidates.parquet', index=False)

del tier1_tp_desc, prot_keyword_index, tier1_transport, transport_proteins_df
gc.collect()

Unmapped transport reactions with names: 4,572


Transport proteins for matching: 16,332



No matches found with >=2 keyword overlap. Transport reaction names may be too generic.


0

## 5. Characterizing Unmapped Non-Transport Reactions

In [9]:
user_ec = pd.read_csv(f'{USER_DIR}/Unique_ModelSEED_Reaction_ECs.txt', sep='\t')
user_ec.columns = [c.strip() for c in user_ec.columns]
rxns_with_ec = set(user_ec['ModelSEED ID'].str.strip())
print(f'Reactions with EC assignment (user lookup): {len(rxns_with_ec):,}')

all_tier_ecs = set()
for parq in ['uniprot_native_protein_ec.parquet', 'pangenome_gc_ec.parquet',
             'curated_evidence_ec.parquet']:
    try:
        ecs = set(pd.read_parquet(f'{DATA_DIR}/{parq}', columns=['ec'])['ec'])
        all_tier_ecs |= ecs
        print(f'  {parq}: {len(ecs):,} unique ECs')
    except Exception:
        pass
rast_ecs = set(pd.read_parquet(f'{DATA_DIR}/rast_protein_ec.parquet', columns=['ec'])['ec'])
all_tier_ecs |= rast_ecs
print(f'  rast: {len(rast_ecs):,} unique ECs')
print(f'Combined unique ECs from all sources: {len(all_tier_ecs):,}')
del rast_ecs
gc.collect()

ec_bridge = pd.read_parquet(f'{DATA_DIR}/ec_to_reaction.parquet')
ec_to_rxn = dict()
for _, row in ec_bridge.iterrows():
    ec_to_rxn.setdefault(row['rxn_bare'], set()).add(row['ec'])

unmapped_nt = merged[(~merged['is_transport']) & (~merged['any_evidence'])].copy()
print(f'\nUnmapped non-transport reactions: {len(unmapped_nt):,}')

def classify_ec_gap(rxn_bare):
    if rxn_bare not in rxns_with_ec:
        return 'No EC assigned'
    rxn_ecs = ec_to_rxn.get(rxn_bare, set())
    if not rxn_ecs:
        return 'EC exists but not in balanced bridge'
    if rxn_ecs & all_tier_ecs:
        return 'EC annotated but not mapped (bridge gap)'
    else:
        return 'EC exists, no protein annotated'

unmapped_nt['ec_status'] = unmapped_nt['rxn_bare'].apply(classify_ec_gap)
print(f'\nEC gap decomposition:')
for status, count in unmapped_nt['ec_status'].value_counts().items():
    print(f'  {status}: {count:,} ({100*count/len(unmapped_nt):.1f}%)')

Reactions with EC assignment (user lookup): 25,756


  uniprot_native_protein_ec.parquet: 5,032 unique ECs


  pangenome_gc_ec.parquet: 3,783 unique ECs
  curated_evidence_ec.parquet: 4,780 unique ECs


  rast: 2,439 unique ECs
Combined unique ECs from all sources: 5,428



Unmapped non-transport reactions: 12,038

EC gap decomposition:
  No EC assigned: 10,832 (90.0%)
  EC exists, no protein annotated: 1,206 (10.0%)


In [10]:
def classify_origin(abbr):
    if pd.isna(abbr):
        return 'No abbreviation'
    if re.match(r'^R\d{5}$', abbr):
        return 'KEGG'
    if re.match(r'^\d+\.\d+\.\d+', abbr):
        return 'EC-derived'
    if re.match(r'^RXN-\d+', abbr) or re.match(r'^[A-Z].*RXN', abbr):
        return 'MetaCyc'
    return 'Other/ModelSEED'

merged['origin'] = merged['abbreviation'].apply(classify_origin)

origin_ct = pd.crosstab(merged['origin'], merged['mapped'], margins=True)
print('Database origin x mapping status:')
print(origin_ct.to_string())

print(f'\nUnmapped non-transport by origin:')
unmapped_nt['origin'] = unmapped_nt['abbreviation'].apply(classify_origin)
for origin, count in unmapped_nt['origin'].value_counts().items():
    print(f'  {origin}: {count:,} ({100*count/len(unmapped_nt):.1f}%)')

Database origin x mapping status:
mapped           Mapped  Unmapped    All
origin                                  
EC-derived         1397        90   1487
KEGG               5865       986   6851
MetaCyc            5237      2205   7442
No abbreviation    1172      4784   5956
Other/ModelSEED    3680      8927  12607
All               17351     16992  34343

Unmapped non-transport by origin:
  Other/ModelSEED: 5,488 (45.6%)
  No abbreviation: 4,402 (36.6%)
  MetaCyc: 1,074 (8.9%)
  KEGG: 986 (8.2%)
  EC-derived: 88 (0.7%)


In [11]:
unmapped_nt_named = unmapped_nt[unmapped_nt['name'].notna()]
print(f'Unmapped non-transport with names: {len(unmapped_nt_named):,} / {len(unmapped_nt):,}')

all_words = []
for name in unmapped_nt_named['name']:
    tokens = re.findall(r'[a-zA-Z][-a-zA-Z]*', name.lower())
    all_words.extend(t for t in tokens if len(t) > 3 and t not in {
        'reaction', 'acid', 'with', 'from', 'into', 'that', 'this', 'the'
    })

word_counts = pd.Series(all_words).value_counts()
print(f'\nTop 30 keywords in unmapped non-transport reaction names:')
for word, count in word_counts.head(30).items():
    print(f'  {word}: {count}')

Unmapped non-transport with names: 7,636 / 12,038

Top 30 keywords in unmapped non-transport reaction names:
  rxn-: 862
  oxidoreductase: 458
  synthase: 205
  oxygen: 195
  nadph: 142
  nadp: 136
  dehydrogenase: 117
  phosphate: 117
  ligase: 112
  reductase: 102
  protein: 79
  synthesis: 68
  hydrolase: 64
  hydroxylase: 64
  hydroxylating: 62
  fatty: 62
  methyl-: 62
  acyl-carrier: 61
  cis-delta: 59
  aminotransferase: 57
  metaexp: 56
  transferase: 48
  s-adenosyl-l-methionine: 48
  monooxygenase: 47
  glucosyltransferase: 46
  nadh: 45
  phosphotransferase: 44
  dehydratase: 43
  oxidase: 43
  decarboxylating: 42


In [12]:
merged['deltag_num'] = pd.to_numeric(merged['deltag'], errors='coerce')
has_deltag = merged[merged['deltag_num'].notna()]

print(f'Reactions with deltag values: {len(has_deltag):,} / {len(merged):,}')
print(f'\nDelta-G statistics (kcal/mol):')
for label, mask in [('Mapped', has_deltag['any_evidence']),
                     ('Unmapped', ~has_deltag['any_evidence'])]:
    subset = has_deltag[mask]['deltag_num']
    print(f'  {label}: mean={subset.mean():.1f}, median={subset.median():.1f}, '
          f'std={subset.std():.1f}, n={len(subset):,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
bins = np.linspace(-200, 200, 80)
ax.hist(has_deltag[has_deltag['any_evidence']]['deltag_num'].clip(-200, 200),
        bins=bins, alpha=0.6, label='Mapped', color='#4c78a8', density=True)
ax.hist(has_deltag[~has_deltag['any_evidence']]['deltag_num'].clip(-200, 200),
        bins=bins, alpha=0.6, label='Unmapped', color='#e45756', density=True)
ax.set_xlabel('Delta-G (kcal/mol)')
ax.set_ylabel('Density')
ax.set_title('Thermodynamic Profile')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
unmapped_origins = unmapped_nt['origin'].value_counts()
mapped_nt_origins = merged[(~merged['is_transport']) & (merged['any_evidence'])]['origin'].value_counts()
origins_all = sorted(set(unmapped_origins.index) | set(mapped_nt_origins.index))
x = np.arange(len(origins_all))
w = 0.35
ax.bar(x - w/2, [mapped_nt_origins.get(o, 0) for o in origins_all], w,
       label='Mapped', color='#4c78a8')
ax.bar(x + w/2, [unmapped_origins.get(o, 0) for o in origins_all], w,
       label='Unmapped', color='#e45756')
ax.set_xticks(x)
ax.set_xticklabels(origins_all, rotation=30, ha='right')
ax.set_ylabel('Non-Transport Reactions')
ax.set_title('Database Origin')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Characterization of Unmapped Non-Transport Reactions', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/unmapped_characterization.png')
plt.show()
print('Saved: unmapped_characterization.png')

Reactions with deltag values: 34,343 / 34,343

Delta-G statistics (kcal/mol):
  Mapped: mean=2689747.6, median=-1.0, std=4434408.2, n=17,351
  Unmapped: mean=5331327.2, median=10000000.0, std=4989162.1, n=16,992


Saved: unmapped_characterization.png


## 6. Summary

In [13]:
print('=' * 65)
print('NB10 TRANSPORT & UNMAPPED REACTION ANALYSIS SUMMARY')
print('=' * 65)

print(f'\n--- Transport Breakdown ---')
print(f'Transport reactions: {total_transport:,} / {len(merged):,} balanced ({100*total_transport/len(merged):.1f}%)')
print(f'Mapped transport:    {mapped_transport:,} / {total_transport:,} ({100*mapped_transport/total_transport:.1f}%)')
print(f'Unmapped transport:  {unmapped_transport:,} / {total_transport:,} ({100*unmapped_transport/total_transport:.1f}%)')
print(f'Transport as % of all unmapped: {100*unmapped_transport/unmapped_total:.1f}%')

n_matches = len(match_df) if len(match_df) > 0 else 0
n_rxns_matched = match_df['rxn_bare'].nunique() if n_matches > 0 else 0
print(f'\n--- Text-Based Matching ---')
print(f'Candidate matches (>=2 keyword overlap): {n_matches:,}')
print(f'Unmapped transport reactions with candidates: {n_rxns_matched:,} / {len(unmapped_t):,}')

print(f'\n--- Unmapped Non-Transport Decomposition ---')
print(f'Total unmapped non-transport: {len(unmapped_nt):,}')
for status, count in unmapped_nt['ec_status'].value_counts().items():
    print(f'  {status}: {count:,} ({100*count/len(unmapped_nt):.1f}%)')

print(f'\nFigures saved:')
print(f'  {FIG_DIR}/transport_breakdown.png')
print(f'  {FIG_DIR}/unmapped_characterization.png')

del merged, unmapped_nt, unmapped_t, match_df, ec_bridge
gc.collect()

NB10 TRANSPORT & UNMAPPED REACTION ANALYSIS SUMMARY

--- Transport Breakdown ---
Transport reactions: 6,004 / 34,343 balanced (17.5%)
Mapped transport:    1,050 / 6,004 (17.5%)
Unmapped transport:  4,954 / 6,004 (82.5%)
Transport as % of all unmapped: 29.2%

--- Text-Based Matching ---
Candidate matches (>=2 keyword overlap): 0
Unmapped transport reactions with candidates: 0 / 4,572

--- Unmapped Non-Transport Decomposition ---
Total unmapped non-transport: 12,038
  No EC assigned: 10,832 (90.0%)
  EC exists, no protein annotated: 1,206 (10.0%)

Figures saved:
  ../figures/transport_breakdown.png
  ../figures/unmapped_characterization.png


53